In [1]:
import pandas as pd
import requests
from datetime import datetime
import openpyxl
from openpyxl.styles import Alignment, Font

# Hàm khôi phục văn bản Abstract từ Inverted Index của OpenAlex
def get_abstract(inv_index):
    if not inv_index:
        return None
    word_list = []
    for word, positions in inv_index.items():
        for pos in positions:
            word_list.append((pos, word))
    word_list.sort(key=lambda x: x[0])
    return " ".join([w[1] for w in word_list])

records = []
page = 1

# Lặp cho đến khi thu thập đủ 50 hồ sơ hoàn toàn sạch
while len(records) < 50 and page <= 20:
    url = f"https://api.openalex.org/works?filter=has_doi:true,has_abstract:true,has_fulltext:true&per_page=50&page={page}"
    response = requests.get(url)
    data = response.json()
    results = data.get('results', [])

    for item in results:
        # 1. Trích xuất Tác giả & Mã ORCID thật
        authorships = item.get('authorships', [])
        authors_list = []
        orcids_list = []

        for a in authorships:
            author_name = a.get('author', {}).get('display_name')
            orcid_url = a.get('author', {}).get('orcid')
            if author_name and orcid_url:
                authors_list.append(author_name)
                orcids_list.append(orcid_url.split('/')[-1]) # Lấy chuỗi mã 0000-000X-XXXX-XXXX

        # Bắt buộc phải có tác giả VÀ mã ORCID đầy đủ
        if not authors_list or not orcids_list or len(authors_list) != len(orcids_list):
            continue

        # 2. Trích xuất các trường dữ liệu còn lại
        title = item.get('title')

        doi_raw = item.get('doi')
        doi = doi_raw.replace('https://doi.org/', '') if doi_raw else None

        publish_year = item.get('publication_year')

        primary_location = item.get('primary_location') or {}
        source = primary_location.get('source') or {}
        venue = source.get('display_name')

        primary_topic = item.get('primary_topic') or {}
        field = primary_topic.get('field', {}).get('display_name') or primary_topic.get('domain', {}).get('display_name')

        concepts = item.get('concepts', [])
        keywords_list = [c.get('display_name') for c in concepts[:5] if c.get('display_name')]
        keywords = ", ".join(keywords_list) if keywords_list else None

        abstract = get_abstract(item.get('abstract_inverted_index'))

        open_access = item.get('open_access') or {}
        pdf_url = open_access.get('oa_url') or primary_location.get('pdf_url')

        source_type_raw = item.get('type')
        if source_type_raw in ['article', 'journal-article']:
            source_type = 'journal'
        elif source_type_raw in ['proceedings-article', 'conference-paper']:
            source_type = 'conference'
        else:
            source_type = 'preprint'

        # 3. ĐIỀU KIỆN LỌC SẠCH KHẮT KHE: Bắt buộc TẤT CẢ các trường phải có dữ liệu
        if all([title, doi, authors_list, orcids_list, publish_year, venue, field, keywords, abstract, pdf_url]):
            records.append({
                "id": len(records) + 1,
                "title": title,
                "doi": doi,
                "authors": "; ".join(authors_list),
                "author_orcids": "; ".join(orcids_list),
                "publish_year": publish_year,
                "venue": venue,
                "field": field,
                "keywords": keywords,
                "abstract": abstract,
                "pdf_url": pdf_url,
                "source_type": source_type,
                "created_at": datetime.now().strftime("%Y-%m-%d")
            })

            if len(records) == 50:
                break
    page += 1

# Xuất dữ liệu ra Excel & Căn chỉnh giao diện
df = pd.DataFrame(records)

file_name = "research_profiles.xlsx"
with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
    df.to_excel(writer, index=False, sheet_name='research_profile')
    worksheet = writer.sheets['research_profile']

    # In đậm tiêu đề
    for cell in worksheet[1]:
        cell.font = Font(bold=True)

    # Tự động dãn cột & bật ngắt dòng
    for col in worksheet.columns:
        col_letter = col[0].column_letter
        max_len = max(len(str(cell.value or '')) for cell in col)

        if col_letter in ['A', 'E', 'F', 'L', 'M']:
            adjusted_width = max(max_len + 3, 12)
        else:
            adjusted_width = min(max(max_len + 3, 15), 45)

        worksheet.column_dimensions[col_letter].width = adjusted_width
        for cell in col:
            cell.alignment = Alignment(vertical='top', wrap_text=True)

print(f"Đã cào {len(records)} hồ sơ")

Đã cào 50 hồ sơ


In [2]:
import sqlite3
import pandas as pd

# 1. Đọc file Excel 50 hồ sơ
df = pd.read_excel("research_profiles.xlsx")

# 2. Tạo một database SQLite nằm ngay trong Colab có tên là "techlinkvn.db"
conn = sqlite3.connect("techlinkvn.db")

# 3. Đẩy toàn bộ dữ liệu từ Pandas DataFrame vào bảng SQL có tên là "papers"
df.to_sql("papers", conn, if_exists="replace", index=False)

# 4. Thử viết một câu lệnh SQL để truy vấn dữ liệu trực tiếp trong Colab luôn!
query = "SELECT title, venue, field FROM papers WHERE field = 'Vật liệu mới' LIMIT 3"
result_df = pd.read_sql(query, conn)

print("--- KẾT QUẢ TRUY VẤN SQL TRÊN COLAB ---")
print(result_df)
# Đóng kết nối
conn.close()

--- KẾT QUẢ TRUY VẤN SQL TRÊN COLAB ---
Empty DataFrame
Columns: [title, venue, field]
Index: []


In [3]:
import sqlite3
import pandas as pd
from IPython.display import display

# 1. Mở kết nối đến database SQLite
conn = sqlite3.connect("techlinkvn.db")

# 2. Lấy toàn bộ dữ liệu từ bảng papers
query = "SELECT * FROM papers"
result_df = pd.read_sql(query, conn)

# 3. Đánh lại thứ tự
result_df.index = range(1, len(result_df) + 1)

# 4. Cấu hình để bảng hiển thị rộng hết cỡ, không bị thu gọn hay cắt bớt cột
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

print("--- TOÀN BỘ 50 BÀI BÁO ---")
display(result_df)

# Đóng kết nối
conn.close()

--- TOÀN BỘ 50 BÀI BÁO ---


,id,title,doi,authors,author_orcids,publish_year,venue,field,keywords,abstract,pdf_url,source_type,created_at
1,1,Radiation Resistant Camera System for Monitoring Deuterium Plasma Discharges in the Large Helical Device,10.1585/pfr.15.2402039,M. Shoji,0000-0003-0655-7347,2020,Plasma and Fusion Research,Physics and Astronomy,"Radiation, Plasma, Materials science, Optics, Shield","Radiation resistant camera system was constructed for monitoring deuterium plasma discharges in the Large Helical Device (LHD). This system has contributed to safe operation during two experimental campaigns without serious problems due to radiation (neutrons and gamma-rays). The cameras steadily functioned even in the plasma discharge with the maximum neutron emission rate in FY 2017, though some bright specks temporarily appeared on the images. The cameras have been installed in shield boxes which consist of lead boxes covered with 10% borated polyethylene blocks in all directions. For optimizing the design of the shield box, the radiation flux distribution was calculated by MCNP-6 code, which reveals the reduction of the radiation flux and the change of the energy spectra in the shield box. Thanks to the optimization, significant extension of the lifetime of the cameras has been realized. Investigation of the influence of the radiation on the CCD image sensor shows that the number of bright specks generally increases with the radiation flux to the camera, which also indicates that some bright specks disappear by the self-annealing process on the image sensor. This phenomenon also highly contributes to the further extension of the lifetime of the radiation resistant cameras.",https://www.jstage.jst.go.jp/article/pfr/15/0/15_2402039/_pdf,journal,2026-09-14
2,2,Using thematic analysis in psychology,10.1191/1478088706qp063oa,Virginia Braun; Victoria Clarke,0000-0002-3435-091X; 0000-0001-9405-7363,2006,Qualitative Research in Psychology,Health Professions,"Thematic analysis, Thematic map, Qualitative analysis, Qualitative research, Relation (database)","Thematic analysis is a poorly demarcated, rarely acknowledged, yet widely used qualitative analytic method within psychology. In this paper, we argue that it offers an accessible and theoretically flexible approach to analysing qualitative data. We outline what thematic analysis is, locating it in relation to other qualitative analytic methods that search for themes or patterns, and in relation to different epistemological and ontological positions. We then provide clear guidelines to those wanting to start thematic analysis, or conduct it in a more deliberate and rigorous way, and consider potential pitfalls in conducting thematic analysis. Finally, we outline the disadvantages and advantages of thematic analysis. We conclude by advocating thematic analysis as a useful and flexible method for qualitative research in and beyond psychology.",https://www.tandfonline.com/doi/abs/10.1191/1478088706qp063oa,journal,2026-09-14
3,3,The PRISMA 2020 statement: an updated guideline for reporting systematic reviews,10.1136/bmj.n71,Matthew J. Page; Joanne E. McKenzie; Patrick M. Bossuyt; Isabelle Boutron; Tammy Hoffmann; Cynthia D. Mulrow; Larissa Shamseer; Jennifer Tetzlaff; Elie A. Akl; Sue Brennan; Roger Chou; Julie Glanville; Jeremy Grimshaw; Asbjørn Hróbjartsson; Manoj M. Lalu; Tianjing Li; Elizabeth Loder; Evan Mayo‐Wilson; Steve McDonald; Luke A. McGuinness; Lesley Stewart; James Thomas; Andrea C. Tricco; Vivian Welch; Penny Whiting; David Moher,0000-0002-4242-7526; 0000-0003-3534-1641; 0000-0003-4427-0128; 0000-0002-5263-6241; 0000-0001-5210-8548; 0000-0002-4768-4492; 0000-0003-3690-3378; 0000-0003-0787-5363; 0000-0002-3444-8618; 0000-0003-1789-8809; 0000-0001-9889-8610; 0000-0002-1253-8524; 0000-0001-8015-8243; 0000-0002-2451-5012; 0000-0002-0322-382X; 0000-0001-5371-4558; 0000-0003-1501-2947; 0000-0001-6126-2459; 0000-0003-2832-5205; 0000-0001-8730-9761; 0000-0003-0287-4724; 0000-0003-4805-4190; 0000-0002-411